OOD Detection Module

In [1]:
import scanpy as sc
from ood_detector import AEOutlierDetector, Annomaly_detector
from sklearn.model_selection import train_test_split
import numpy as np

In [2]:
filtered_data_file = "data/LCA_TS/LCA_TS.h5ad"
alpha_OOD = 0.3
delta_OOD = 0.1
network_architecture_OOD = {
    "hidden_sizes": [50, 48, 32, 24], 
    "dropout_rates": [0.15, 0.15, 0.15, 0.15],
    "learning_rate": 1e-4,
    "batch_size": 72,
    "n_epochs": 1000,
    "patience":      9,
    "noise_level":   0.1,
    "lambda_sparse": 1e-3}
adata = sc.read_h5ad(filtered_data_file)
adata

AnnData object with n_obs × n_vars = 16373 × 24204
    obs: 'free_annotation', 'donor_id', 'compartment', 'cell_type_ontology_term_id', 'cell_type', 'assay', 'sex', 'tissue', 'split'

In [4]:
sc.pp.normalize_total(adata,target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
adata = adata[:, adata.var.highly_variable]

#adata.X = adata.X.toarray()
data_train_val = adata[adata.obs['split']=='train']

indices = data_train_val.obs_names
labels = data_train_val.obs['cell_type']

train_idx, val_idx = train_test_split(
    indices, 
    test_size=0.2,          
    stratify=labels,
    random_state=42
)
data_train_OOD = adata[train_idx,:].X.toarray().astype(np.float32)
data_val_OOD = adata[val_idx,:].X.toarray().astype(np.float32)
data_cal_OOD = adata[adata.obs['split']=='calibrate'].X.toarray().astype(np.float32)
data_OOD = adata[adata.obs['split']=='test'].X.toarray().astype(np.float32)

/root/anaconda3/envs/zxr_hcc1/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


In [6]:
model_oc = AEOutlierDetector(input_dim=data_train_OOD.shape[1], network_architecture=network_architecture_OOD)
        
OOD_detector = Annomaly_detector("marginal", oc_model = model_oc, delta=delta_OOD)

OOD_detector.fit(data_train_OOD, data_val_OOD, data_cal_OOD)

Using device: cuda


/root/anaconda3/envs/zxr_hcc1/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Using device: cuda
Epoch 1, Training Loss: 0.2043, Validation Loss: 0.1828
Epoch 2, Training Loss: 0.1965, Validation Loss: 0.1750
Epoch 3, Training Loss: 0.1908, Validation Loss: 0.1703
Epoch 4, Training Loss: 0.1864, Validation Loss: 0.1665
Epoch 5, Training Loss: 0.1829, Validation Loss: 0.1633
Epoch 6, Training Loss: 0.1795, Validation Loss: 0.1603
Epoch 7, Training Loss: 0.1766, Validation Loss: 0.1578
Epoch 8, Training Loss: 0.1737, Validation Loss: 0.1555
Epoch 9, Training Loss: 0.1713, Validation Loss: 0.1533
Epoch 10, Training Loss: 0.1689, Validation Loss: 0.1513
Epoch 11, Training Loss: 0.1666, Validation Loss: 0.1495
Epoch 12, Training Loss: 0.1647, Validation Loss: 0.1479
Epoch 13, Training Loss: 0.1626, Validation Loss: 0.1463
Epoch 14, Training Loss: 0.1609, Validation Loss: 0.1449
Epoch 15, Training Loss: 0.1590, Validation Loss: 0.1436
Epoch 16, Training Loss: 0.1575, Validation Loss: 0.1423
Epoch 17, Training Loss: 0.1558, Validation Loss: 0.1411
Epoch 18, Training Lo

In [7]:
OOD_detector.predict_pvalues(data_OOD,  X_ID=None)
data_OOD_mask = OOD_detector.evaluate_marginal_pvalues(alpha=alpha_OOD, lambda_par=0.15, use_sbh=True, is_outlier = None)
#data_OOD_mask, accuracy_OOD, precision_OOD, recall_OOD, auroc_OOD = OOD_detector.evaluate_marginal_pvalues(alpha=alpha_OOD, lambda_par=0.15, use_sbh=True, is_outlier = None)

Effective alpha: 0.3108 (alpha=0.3000)


In [8]:
sum(data_OOD_mask)

1938

In [9]:
import pandas as pd
df_ood = pd.DataFrame({'OOD': data_OOD_mask}, index=adata[adata.obs['split'] == 'test'].obs_names)

In [10]:
df_ood.to_csv('LCA_TS_OOD_0.3.txt', sep='\t', index=True)